# Validate Sentinel-1 GRD RTC GeoZarr stores

Quick, repeatable correctness check for a single `s1-grd-rtc-{tile}.zarr` store
produced by the S1 GRD ingest pipeline. It runs three layers of validation and
prints a clear **PASS / WARN / FAIL** summary at the end:

1. **Structural** — group/array hierarchy via the `S1RtcRoot` pydantic-zarr model.
2. **Metadata / CF** — CRS (`proj:code`), GeoZarr conventions, dims & dtypes,
   and whether `grid_mapping` is wired so rioxarray / TiTiler can read it.
3. **Data sanity + visual** — NaN fractions, backscatter value ranges, overview
   pyramid consistency, plus VV/VH maps, a SAR false-colour RGB, `border_mask`,
   and dB histograms.

**How to use:** edit the `STORE_PATH` (and optionally `ORBIT`) in the config cell,
then *Run All*. The default points at the bundled local test fixture so the
notebook runs offline. `STORE_PATH` also accepts a remote HTTPS URL — e.g. a STAC
item's `zarr-store` asset href — for validating published tiles directly.

> Severity model: a **WARN** is informational (e.g. an older store missing the
> CF `spatial_ref` wiring) and does **not** fail the store; only a **FAIL** does.


## ⚙️ Configuration

In [ ]:
from pathlib import Path


def _repo_root() -> Path:
    p = Path.cwd().resolve()
    for cand in (p, *p.parents):
        if (cand / "pyproject.toml").exists():
            return cand
    return p


REPO_ROOT = _repo_root()

# Point STORE_PATH at the GeoZarr store to validate.
#   • Local default : bundled S1 GRD RTC test fixture (runs offline).
#   • S3 staging     : STORE_PATH = "s3://<bucket>/s1-grd-rtc-<tile>.zarr"
#                      and export AWS_ENDPOINT_URL (+ AWS creds) before launch.
STORE_PATH = str(
    REPO_ROOT / "s3:/esa-zarr-sentinel-explorer-tests/s1-rtc-test/s1-grd-rtc-31TCH.zarr"
)
# Remote store? Point STORE_PATH at an HTTPS URL — e.g. a STAC item's
# `zarr-store` asset href. Public EOPF test stores need no credentials:
#   STORE_PATH = ("https://s3.explorer.eopf.copernicus.eu/esa-zarr-sentinel-explorer-tests"
#                 "/sentinel-1-grd-rtc-tests/s1-grd-rtc-31TDH.zarr")
# STORAGE_OPTIONS passes fsspec kwargs through to the store backend
# (e.g. {"anon": True}, or endpoint_url / key / secret for an s3:// path).
STORAGE_OPTIONS: dict = {}

ORBIT = None  # None -> auto-detect the first orbit group present
RES_FOR_PLOTS = "r60m"  # overview level used for full-scene plots / data checks
TIME_INDEX = 0  # which acquisition (time step) to visualise

IS_REMOTE = "://" in STORE_PATH
print("STORE_PATH    =", STORE_PATH)
print("location      =", "remote" if IS_REMOTE else "local")
print("ORBIT         =", ORBIT or "(auto-detect)")
print("RES_FOR_PLOTS =", RES_FOR_PLOTS)
print("TIME_INDEX    =", TIME_INDEX)

## 📦 Imports & helpers

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray  # noqa: F401  -- registers the .rio accessor on xarray objects
import xarray as xr
import zarr
from eopf_geozarr.data_api.s1_rtc import REQUIRED_CONVENTION_UUIDS, S1RtcRoot
from pydantic import ValidationError
from pyproj import CRS
from zarr.storage import FsspecStore

warnings.filterwarnings("ignore")  # silence Zarr V3 unstable-dtype / consolidation notices

# ── store access ────────────────────────────────────────────────────────────
# Works for local stores and for remote (https / s3) consolidated stores.
# Levels are always opened from the store *root* via group= so that consolidated
# metadata (held at the root of production stores) is used — this avoids object
# listing, which public HTTPS endpoints typically disallow.
_SO = STORAGE_OPTIONS or None


def open_root():
    if IS_REMOTE:
        return zarr.open_group(FsspecStore.from_url(STORE_PATH, storage_options=_SO), mode="r")
    return zarr.open_group(STORE_PATH, mode="r", zarr_format=3)


def open_level(orbit, res, decode_coords="all"):
    return xr.open_zarr(
        STORE_PATH,
        group=f"{orbit}/{res}",
        consolidated=None,
        decode_coords=decode_coords,
        storage_options=_SO,
    )


# ── check accumulator ───────────────────────────────────────────────────────
# Each entry is (name, severity, detail); severity in {PASS, WARN, FAIL}.
# Overall result = FAIL if any FAIL, else PASS (warnings are informational).
_ICONS = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌"}
checks: list[tuple[str, str, str]] = []


def record(name: str, severity: str, detail: str = "") -> str:
    checks.append((name, severity, detail))
    print(f"{_ICONS[severity]} {name}" + (f" — {detail}" if detail else ""))
    return severity


def ok(name, detail=""):
    return record(name, "PASS", detail)


def warn(name, detail=""):
    return record(name, "WARN", detail)


def fail(name, detail=""):
    return record(name, "FAIL", detail)

## 1 · Open store & detect orbit / resolution levels
Opens the root group, picks the orbit direction (auto-detected unless `ORBIT` is
set), and lists the multiscale resolution levels and the optional `conditions/`
ancillary group.

In [ ]:
root = open_root()
root_members = list(root)
orbit_groups = [m for m in ("ascending", "descending") if m in root_members]

if not orbit_groups:
    fail("Store has an orbit group", f"no ascending/descending in {root_members}")
    raise SystemExit("No orbit group found — cannot continue.")
ok("Store opens", f"root members: {root_members}")

orbit = ORBIT or orbit_groups[0]
if orbit not in orbit_groups:
    raise SystemExit(f"Requested ORBIT={orbit!r} not present (have {orbit_groups})")
print(f"\nValidating orbit group: {orbit!r}")

orbit_grp = root[orbit]
orbit_members = list(orbit_grp)
levels = [m for m in orbit_members if m.startswith("r") and m.endswith("m") and m[1:-1].isdigit()]
levels = sorted(levels, key=lambda r: int(r[1:-1]))
has_conditions = "conditions" in orbit_members

print("Resolution levels:", levels)
print("Conditions group :", "present" if has_conditions else "absent")

if RES_FOR_PLOTS not in levels:
    warn(
        "RES_FOR_PLOTS available",
        f"{RES_FOR_PLOTS!r} not in {levels}; falling back to {levels[-1]!r}",
    )
    RES_FOR_PLOTS = levels[-1]

## 2 · Structural validation — `S1RtcRoot`
Validates the full hierarchy against the pydantic-zarr `S1RtcRoot` model.

> **Known model drift:** the bundled model is *closed* and does not list the
> `x`/`y`/`spatial_ref` coordinate arrays that the current ingest writes at every
> level. Those specific `extra_forbidden` errors are filtered out and reported as
> a single WARN; any *other* structural error is a real FAIL.

In [ ]:
_KNOWN_EXTRA = ("x", "y", "spatial_ref")
try:
    S1RtcRoot.from_zarr(root)
    ok("Strict schema (S1RtcRoot)", "validates with no deviations")
except ValidationError as exc:
    errs = exc.errors()
    drift = [
        e
        for e in errs
        if e["type"] == "extra_forbidden" and e["loc"] and e["loc"][-1] in _KNOWN_EXTRA
    ]
    real = [e for e in errs if e not in drift]
    if real:
        detail = "; ".join("/".join(map(str, e["loc"])) + f": {e['type']}" for e in real[:5])
        fail("Strict schema (S1RtcRoot)", f"{len(real)} real error(s): {detail}")
    else:
        warn(
            "Strict schema (S1RtcRoot)",
            f"only known x/y/spatial_ref coordinate-array drift ({len(drift)} entries) — OK",
        )

## 3 · Metadata / CF checks
Authoritative CRS comes from the orbit group's `proj:code`. We also confirm the
GeoZarr convention UUIDs, the native `r10m` dims/dtypes and per-acquisition
coordinates, and whether `grid_mapping` is wired so rioxarray/TiTiler resolve a
CRS (absence is a WARN — older stores need the CF patch).

In [ ]:
attrs = dict(orbit_grp.attrs)

# proj:code -> authoritative CRS
proj_code = attrs.get("proj:code")
try:
    crs = CRS.from_user_input(proj_code)
    ok("CRS (proj:code)", f"{proj_code} → {crs.name}")
except Exception as e:  # noqa: BLE001
    crs = None
    fail("CRS (proj:code)", f"{proj_code!r} not parseable: {e}")

for key in ("spatial:bbox", "spatial:dimensions", "multiscales"):
    (ok if key in attrs else fail)(f"orbit attr '{key}'", "present" if key in attrs else "missing")

conv = attrs.get("zarr_conventions", [])
conv_uuids = {c.get("uuid") for c in conv} if isinstance(conv, list) else set()
missing_uuids = set(REQUIRED_CONVENTION_UUIDS) - conv_uuids
(ok if not missing_uuids else fail)(
    "zarr_conventions UUIDs",
    "all required present" if not missing_uuids else f"missing {sorted(missing_uuids)}",
)

In [ ]:
# Native r10m: dims, dtypes, per-acquisition coordinates, grid_mapping wiring.
native = open_level(orbit, "r10m")

expected_dtype = {"vv": "float32", "vh": "float32", "border_mask": "uint8"}
for var, dt in expected_dtype.items():
    if var not in native:
        fail(f"r10m '{var}' present", "missing")
        continue
    got = str(native[var].dtype)
    (ok if got == dt else fail)(f"r10m '{var}' dtype", f"{got} (want {dt})")
    dims = tuple(native[var].dims)
    (ok if dims == ("time", "y", "x") else fail)(f"r10m '{var}' dims", str(dims))

for c in ("time", "relative_orbit", "absolute_orbit", "platform"):
    (ok if c in native.variables else warn)(
        f"r10m '{c}'", "present" if c in native.variables else "missing"
    )

rio_crs = native["vv"].rio.crs if "vv" in native else None
if rio_crs is not None:
    ok("grid_mapping wired (.rio.crs)", str(rio_crs))
else:
    warn(
        "grid_mapping wired (.rio.crs)",
        "rioxarray could not resolve a CRS — store lacks spatial_ref; "
        "TiTiler needs the CF grid_mapping patch",
    )

## 4 · Data-sanity checks
Read at `RES_FOR_PLOTS` (fast). Checks finite fraction, linear gamma0 ranges and
plausible dB ranges per polarisation, a binary `border_mask`, and that the
overview pyramid shrinks monotonically with a mean consistent with native.

In [ ]:
ds = open_level(orbit, RES_FOR_PLOTS)
nt = ds.sizes["time"] if "time" in ds.dims else 1
ti = min(TIME_INDEX, nt - 1)


def band(name):
    da = ds[name]
    return da.isel(time=ti) if "time" in da.dims else da


for pol in ("vv", "vh"):
    a = band(pol).values.astype("float64")
    finite = np.isfinite(a)
    frac = float(finite.mean())
    if frac == 0.0:
        fail(f"{pol} finite data", "0% finite (all-NaN)")
        continue
    vmin, vmax = float(np.min(a[finite])), float(np.max(a[finite]))
    (ok if frac > 0.01 else warn)(
        f"{pol} finite data", f"{frac:.1%} finite, linear range [{vmin:.3g}, {vmax:.3g}]"
    )
    pos = a[finite & (a > 0)]
    if pos.size:
        db = 10.0 * np.log10(pos)
        p2, p98 = float(np.percentile(db, 2)), float(np.percentile(db, 98))
        plausible = (p2 >= -40.0) and (p98 <= 10.0)
        (ok if plausible else warn)(f"{pol} dB range", f"p2..p98 = [{p2:.1f}, {p98:.1f}] dB")

bm = band("border_mask").values
uniq = np.unique(bm)
(ok if set(uniq.tolist()).issubset({0, 1}) and uniq.size > 1 else warn)(
    "border_mask binary", f"unique values {uniq.tolist()[:6]}"
)

In [ ]:
# Overview pyramid: shapes halve each step; coarse mean tracks the native mean.
shapes, means = {}, {}
for lv in levels:
    d = open_level(orbit, lv)
    v = d["vv"]
    v0 = v.isel(time=ti) if "time" in v.dims else v
    shapes[lv] = (int(d.sizes["y"]), int(d.sizes["x"]))
    means[lv] = float(np.nanmean(v0.values))

pyramid = pd.DataFrame({"y_x_shape": shapes, "vv_mean": means})
print(pyramid.to_string())

shrink_ok = all(shapes[levels[i + 1]][0] < shapes[levels[i]][0] for i in range(len(levels) - 1))
(ok if shrink_ok else fail)(
    "overview shapes decrease", " > ".join(f"{lv}:{shapes[lv][0]}" for lv in levels)
)

if "r10m" in means and len(means) > 1:
    base, coarse = means["r10m"], means[levels[-1]]
    rel = abs(coarse - base) / (abs(base) + 1e-9)
    (ok if rel < 0.5 else warn)(
        "overview mean consistency",
        f"r10m={base:.4g} vs {levels[-1]}={coarse:.4g} (rel diff {rel:.1%})",
    )

## 5 · Visual validation
VV/VH backscatter maps (dB, north-up, percentile-clipped), a SAR false-colour RGB
(R=VV, G=VH, B=VV/VH), the `border_mask`, dB histograms, and — when more than one
acquisition is present — VV across time.

In [ ]:
acq_times = pd.to_datetime(native["time"].values) if "time" in native.variables else None
rel_orbits = native["relative_orbit"].values if "relative_orbit" in native.variables else None
tile_id = Path(STORE_PATH).name.replace("s1-grd-rtc-", "").replace(".zarr", "")


def to_db(da):
    a = da.values.astype("float64")
    out = np.full(a.shape, np.nan)
    m = np.isfinite(a) & (a > 0)
    out[m] = 10.0 * np.log10(a[m])
    return out


def extent_of(d):
    x, y = d["x"].values, d["y"].values
    return [float(x.min()), float(x.max()), float(y.min()), float(y.max())]


ext = extent_of(ds)
_bits = [f"tile {tile_id}", orbit, RES_FOR_PLOTS]
if acq_times is not None:
    _bits.append(str(acq_times[ti].date()))
if rel_orbits is not None:
    _bits.append(f"relorb {int(rel_orbits[ti])}")
suptitle = "  ·  ".join(_bits)

vv_db, vh_db = to_db(band("vv")), to_db(band("vh"))

fig, axes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)
for ax, data, name in zip(axes, (vv_db, vh_db), ("VV", "VH"), strict=False):
    p2, p98 = np.nanpercentile(data, 2), np.nanpercentile(data, 98)
    im = ax.imshow(data, extent=ext, origin="upper", cmap="gray", vmin=p2, vmax=p98)
    ax.set_title(f"{name} (dB)")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    fig.colorbar(im, ax=ax, shrink=0.8, label="dB")
fig.suptitle(suptitle)
plt.show()

In [ ]:
# SAR false-colour composite: R=VV(dB), G=VH(dB), B=VV/VH ratio(dB).
def _norm(a):
    p2, p98 = np.nanpercentile(a, 2), np.nanpercentile(a, 98)
    return np.clip((a - p2) / (p98 - p2 + 1e-9), 0, 1)


ratio = np.full(vv_db.shape, np.nan)
m = np.isfinite(vv_db) & np.isfinite(vh_db)
ratio[m] = vv_db[m] - vh_db[m]
rgb = np.nan_to_num(np.dstack([_norm(vv_db), _norm(vh_db), _norm(ratio)]))

fig, ax = plt.subplots(figsize=(7, 7), constrained_layout=True)
ax.imshow(rgb, extent=ext, origin="upper")
ax.set_title(f"SAR false-colour (R=VV, G=VH, B=VV/VH)\n{suptitle}")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
plt.show()

In [ ]:
# border_mask map + VV/VH dB histograms.
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
im = axes[0].imshow(band("border_mask").values, extent=ext, origin="upper", cmap="viridis")
axes[0].set_title("border_mask")
axes[0].set_xlabel("x [m]")
axes[0].set_ylabel("y [m]")
fig.colorbar(im, ax=axes[0], shrink=0.8)
axes[1].hist(vv_db[np.isfinite(vv_db)].ravel(), bins=80, alpha=0.6, label="VV")
axes[1].hist(vh_db[np.isfinite(vh_db)].ravel(), bins=80, alpha=0.6, label="VH")
axes[1].set_title("Backscatter distribution")
axes[1].set_xlabel("dB")
axes[1].set_ylabel("pixel count")
axes[1].legend()
fig.suptitle(suptitle)
plt.show()

In [ ]:
# VV across acquisitions (if the store holds more than one time step).
if nt > 1:
    fig, axes = plt.subplots(1, nt, figsize=(5 * nt, 5), constrained_layout=True)
    axes = np.atleast_1d(axes)
    for i in range(nt):
        d = to_db(ds["vv"].isel(time=i))
        p2, p98 = np.nanpercentile(d, 2), np.nanpercentile(d, 98)
        axes[i].imshow(d, extent=ext, origin="upper", cmap="gray", vmin=p2, vmax=p98)
        label = str(acq_times[i].date()) if acq_times is not None else f"t{i}"
        axes[i].set_title(f"VV · {label}")
        axes[i].set_xlabel("x [m]")
    fig.suptitle(f"VV across acquisitions  ·  tile {tile_id}  ·  {orbit}")
    plt.show()
else:
    print("Single acquisition — skipping the time small-multiples plot.")

## 6 · Summary

In [ ]:
summary = pd.DataFrame(checks, columns=["check", "severity", "detail"])
summary.insert(0, "status", summary["severity"].map(_ICONS))

n_pass = int((summary.severity == "PASS").sum())
n_warn = int((summary.severity == "WARN").sum())
n_fail = int((summary.severity == "FAIL").sum())
overall = "FAIL" if n_fail else "PASS"

try:
    display(summary[["status", "check", "detail"]])
except NameError:
    print(summary[["status", "check", "detail"]].to_string(index=False))

bar = "=" * 72
print(f"\n{bar}")
print(
    f"{_ICONS[overall]}  OVERALL: {overall}   "
    f"({n_pass} passed · {n_warn} warnings · {n_fail} failed)"
)
print(f"   store : {STORE_PATH}")
print(f"   orbit : {orbit}")
print(bar)